# Homework Option A: Extend Your Agent

Time: up to 60 minutes

This notebook contains the working agent from Part 3.
Look for `# TODO` comments to add your own tools.
See `Homework_OptionA_Agent.md` for full instructions.

In [ ]:
!pip install -q openai

In [ ]:
from google.colab import userdata
import os

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
print('API key loaded from Colab Secrets.')

os.environ['GROQ_API_KEY'] = GROQ_API_KEY

In [ ]:
from openai import OpenAI
import math, json

MODEL = 'openai/gpt-oss-20b'   # check console.groq.com/docs/models for current names

client = OpenAI(
    base_url='https://api.groq.com/openai/v1',
    api_key=GROQ_API_KEY,
)

---

## Existing Tools (from Part 3)

These two tools already work. Use them as a reference for the pattern.

In [ ]:
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'calculate',
            'description': 'Evaluate a math expression. Use for any calculation.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'expression': {'type': 'string', 'description': "A Python math expression, e.g. '2**10'"}
                },
                'required': ['expression']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'lookup',
            'description': 'Look up the definition of a machine learning term.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'term': {'type': 'string', 'description': 'The ML term to look up'}
                },
                'required': ['term']
            }
        }
    },

    # TODO: add your Tool 1 schema here
    # {
    #     'type': 'function',
    #     'function': {
    #         'name': 'your_tool_name',
    #         'description': 'What your tool does.',
    #         'parameters': {
    #             'type': 'object',
    #             'properties': {
    #                 'param_name': {'type': 'string', 'description': '...'}
    #             },
    #             'required': ['param_name']
    #         }
    #     }
    # },

    # TODO: add your Tool 2 schema here
]

def calculate(expression):
    try:
        return str(eval(expression, {'math': math, '__builtins__': {}}))
    except Exception as e:
        return f'Error: {e}'

GLOSSARY = {
    'transformer' : 'A neural network that uses self-attention to process sequences in parallel.',
    'tokenizer'   : 'A component that splits text into tokens and maps them to integer IDs.',
    'embedding'   : 'A dense vector that represents the meaning of a token or sequence.',
    'rag'         : 'Retrieval-Augmented Generation: combining a retriever with an LLM for document-based QA.',
}

def lookup(term):
    return GLOSSARY.get(term.lower(), f"Term '{term}' not found in glossary.")

# TODO: implement your Tool 1 function here
# def your_tool_name(param_name):
#     ...
#     return result

# TODO: implement your Tool 2 function here

print('Tools defined:', [t['function']['name'] for t in tools])

<details>
<summary><b>Click to show a possible solution (tools)</b></summary>

One way to implement two tools: a distance converter and a text counter.
Copy this into the cell above, replacing the TODO sections.

```python
    {
        'type': 'function',
        'function': {
            'name': 'convert_distance',
            'description': 'Convert a distance between kilometers and miles.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'value': {'type': 'number', 'description': 'The distance value to convert'},
                    'direction': {'type': 'string', 'description': "Either 'km_to_miles' or 'miles_to_km'"}
                },
                'required': ['value', 'direction']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'count_text',
            'description': 'Count words or characters in a piece of text.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'text': {'type': 'string', 'description': 'The text to analyze'},
                    'unit': {'type': 'string', 'description': "Either 'words' or 'characters'"}
                },
                'required': ['text', 'unit']
            }
        }
    },
```

And the function implementations:

```python
def convert_distance(value, direction):
    if direction == 'km_to_miles':
        return f'{value} km = {value * 0.621371:.2f} miles'
    elif direction == 'miles_to_km':
        return f'{value} miles = {value * 1.60934:.2f} km'
    return f'Unknown direction: {direction}'

def count_text(text, unit):
    if unit == 'words':
        return f'{len(text.split())} words'
    elif unit == 'characters':
        return f'{len(text)} characters'
    return f'Unknown unit: {unit}'
```

Remember: your own choice of tools is just as valid. This is one example, not the only answer.
</details>

---

## Agent Loop

This is the same pattern from Part 3.
Add a line in the `if/elif` block for each new tool.

In [ ]:
def run_agent(user_message):
    messages = [
        {'role': 'system', 'content': 'You are a helpful assistant. Use tools when needed.'},
        {'role': 'user',   'content': user_message},
    ]

    response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    msg = response.choices[0].message

    if msg.tool_calls:
        messages.append(msg)
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)

            if name == 'calculate':
                result = calculate(**args)
            elif name == 'lookup':
                result = lookup(**args)
            # TODO: add a branch for your Tool 1
            # elif name == 'your_tool_name':
            #     result = your_tool_name(**args)
            # TODO: add a branch for your Tool 2
            else:
                result = 'Unknown tool'

            print(f'  [Tool] {name}({args}) -> {result}')
            messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': result})

        final = client.chat.completions.create(model=MODEL, messages=messages)
        return final.choices[0].message.content

    return msg.content


print('Agent function ready.')

<details>
<summary><b>Click to show a possible solution (routing)</b></summary>

Add these branches to the `if/elif` block inside `run_agent`,
matching the tool names you defined above:

```python
            elif name == 'convert_distance':
                result = convert_distance(**args)
            elif name == 'count_text':
                result = count_text(**args)
```
</details>

---

## Test Your Agent

Write 4 to 6 questions: some that need Tool 1, some that need Tool 2,
and at least one that needs no tool at all.

In [ ]:
# TODO: write your test questions here
test_questions = [
    'What is 2 to the power of 10?',          # uses existing calculate tool, example
    # 'your question that needs Tool 1',
    # 'your question that needs Tool 2',
    'What is the capital of Italy?',          # should need NO tool
]

for q in test_questions:
    print(f'Question : {q}')
    print(f'Answer   : {run_agent(q)}')
    print()

<details>
<summary><b>Click to show example test questions</b></summary>

```python
test_questions = [
    'How many miles is 50 km?',
    'I am driving 26.2 miles for a marathon, how many km is that?',
    'How many words are in this sentence: The quick brown fox jumps over the lazy dog',
    'How many characters are in the word artificial intelligence?',
    'What is the capital of Italy?',   # should need NO tool
]
```

Expected: the distance questions call `convert_distance`, the word/character
questions call `count_text`, and the capital question needs no tool at all.
</details>

---

## Your Report

Answer these questions in this cell:

**1. What two tools did you build?**

_(your answer here)_

**2. For each test question: did the agent pick the correct tool? Was the result correct?**

_(your answer here)_

**3. Did you find any question where the agent picked the wrong tool, or no tool when it should have? What do you think went wrong?**

_(your answer here)_